In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp


### Ising model using tensor networks   

In [2]:
sigmaz=np.array([[1,0],[0,-1]])
N=12
def random_tensor(length,bond_dim):
    tensor=[]
    tensor.append(np.random.randn(1,2,bond_dim))
    for i in range(length-2):
        tensor.append(np.random.randn(bond_dim,2,bond_dim))
    tensor.append(np.random.randn(bond_dim,2,1))
    return tensor

id=np.eye(2)
def single_site_op(op,site):
    result=1
    for i in range(N):
        if i==site:
            result=np.kron(result,op)
        else:
            result=np.kron(result,id)
    return result
def two_site_op(op1,op2,site):

    result=1

    for i in range(N):

        if i==site:
            result=np.kron(result,op1)

        elif i==site+1:
            result=np.kron(result,op2)

        else:
            result=np.kron(result,id)

    return result
H=0
for i in range(N-1):
    H+=two_site_op(sigmaz,sigmaz,i)

def mps_to_state(A):

    state=A[0]

    for tensor in A[1:]:

        state=np.tensordot(
            state,
            tensor,
            axes=([-1],[0])
        )

    return state.reshape(-1)

def energy(A):

    state = mps_to_state(A)

    numerator = state.conj() @ H @ state
    denominator = state.conj() @ state

    return np.real(numerator/denominator)
def optimize(A, iterations=100, step_size=0.01):

    E = energy(A)

    for step in range(iterations):
        if step%100==0:
            print(step, E)

        for site in range(len(A)):

            # remember old tensor
            old_tensor = A[site].copy()

            # random perturbation
            A[site] += step_size*np.random.randn(*A[site].shape)

            new_E = energy(A)

            if new_E < E:

                E = new_E

            else:

                # reject the move
                A[site] = old_tensor

    return A, E

In [3]:
A=random_tensor(N,4)
state=mps_to_state(A)
E=(state.conj()@H@state)/(state.conj()@state)
print(E)

-0.8341945737537614


In [4]:
A = random_tensor(N,4)

A, E = optimize(A, iterations=500)

print(E)

0 0.697830522502822
100 -2.774082749202624
200 -5.25534659878827
300 -6.539827755697805
400 -7.774275784785262
-9.490494697613725
